[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/05_exploration_marl/05_exploration_marl.ipynb)

# 05 · 探索与前沿（纯 numpy，从零）

目标：实现 **count-based** 与 **RND** 内在奖励驱动探索（稀疏奖励下系统铺开）、看两个学习者在 **博弈** 里收敛到混合纳什、并用 **Decision Transformer** 按目标回报条件生成动作。

路线：稀疏奖励的探索失败 → count-based 探索奖励 → RND 内在奖励 → MARL(RPS 纳什) → Decision Transformer return-conditioning → ✏️ 练习(count/RND/MARL/DT) → 📖 答案 → 🧪 真实硬探索胶囊。

> 心智模型：**探索=把『没见过』变成可优化的奖励；MARL=环境不再平稳；DT=把 RL 变成『按目标回报的序列预测』**。

## 1 · 稀疏奖励：ε-贪婪为什么撞不到奖励

一个长链任务：要连续做对 `L` 步才得奖励。`ε`-贪婪靠随机撞对整条链的概率 ≈ `|A|^(-L)`，**随链长指数衰减**。
先用蒙特卡洛量化这个绝望的概率。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def random_reach_prob(chain_len, n_actions=4, trials=200000):
    '''随机策略走 chain_len 步、每步要选对特定动作(设为0)才继续，估撞到奖励的概率。'''
    success = 0
    for _ in range(trials):
        if all(rng.integers(0, n_actions) == 0 for _ in range(chain_len)):
            success += 1
    return success / trials

for L in [1, 3, 5, 7]:
    p_emp = random_reach_prob(L)
    p_theory = 0.25 ** L
    print(f'链长 L={L}: 撞到奖励概率 ≈ {p_emp:.5f} (理论 {p_theory:.5f})')
# 理论概率 |A|^-L 指数衰减
assert 0.25**7 < 0.25**3 < 0.25**1, '概率随链长指数衰减'
print('\n绝望的数学：L=10、|A|=4 时概率≈1e-6，平均要试百万步才撞一次奖励')
print('✅ 稀疏奖励下 ε-贪婪几乎不可能靠运气走到 -> 需要有方向的探索')

## 2 · Count-based 探索：奔向访问最少的地方

内在奖励 `r_int(s) = β/√N(s)`：访问次数 `N(s)` 越少奖励越高，驱动智能体去没去过的地方。
最干净的对照：**随机游走** vs **count 驱动**(每步走向访问最少的邻居)，看后者**覆盖整个网格快得多**。

In [ ]:
class Grid:
    '''7x7 网格，提供邻居查询(上下左右，撞墙留原地)。'''
    def __init__(self, n=7): self.n=n; self.nS=n*n; self.s=0
    def reset(self): self.s=0; return self.s
    def neighbors(self, s):
        r, c = s//self.n, s%self.n; out = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr = min(max(r+dr,0), self.n-1); nc = min(max(c+dc,0), self.n-1)
            out.append(nr*self.n + nc)
        return out

def count_bonus(N, s, beta=1.0):
    return beta / np.sqrt(N[s] + 1e-8)            # 访问越少奖励越高

def coverage_curve(strategy, steps=400, seed=0):
    '''返回每步累计访问过的不同状态数。strategy: random | count。'''
    r = np.random.default_rng(seed); env = Grid(7); s = env.reset()
    N = np.zeros(env.nS); N[s] += 1; visited = {s}; curve = []
    for _ in range(steps):
        nbrs = env.neighbors(s)
        if strategy == 'random':
            s = nbrs[r.integers(0, 4)]
        else:  # count: 走向 count 内在奖励最高(=访问最少)的邻居
            bonuses = np.array([count_bonus(N, nb) for nb in nbrs])
            best = np.flatnonzero(bonuses == bonuses.max())
            s = nbrs[r.choice(best)]
        N[s] += 1; visited.add(s); curve.append(len(visited))
    return curve

def steps_to_cover(curve, target):
    for i, v in enumerate(curve):
        if v >= target: return i
    return len(curve)

cr = coverage_curve('random', 400, seed=0)
cc = coverage_curve('count', 400, seed=0)
print(f'随机游走 400步覆盖 {cr[-1]}/49，覆盖40个状态用了 {steps_to_cover(cr,40)} 步')
print(f'count驱动 400步覆盖 {cc[-1]}/49，覆盖40个状态用了 {steps_to_cover(cc,40)} 步')
assert steps_to_cover(cc, 40) < steps_to_cover(cr, 40), 'count 探索应更快覆盖'
assert cc[-1] >= cr[-1], 'count 探索最终覆盖不少于随机'
print('✅ count-based 探索系统地奔向没去过的地方 -> 覆盖整个空间快 3~5 倍')

## 3 · RND：随机网络蒸馏

**目标网 `f`（固定随机、永不训练）+ 预测网 `g`（训练去拟合 f）**。内在奖励 = `||g(s)-f(s)||²`。
见过的状态 g 学得准→误差低(不新颖)；新状态 g 没学过→误差高(新颖)。验证这个核心性质。

In [ ]:
def init_net(din, h, dout, seed):
    r = np.random.default_rng(seed)
    return [r.normal(0,np.sqrt(2/din),(h,din)), np.zeros(h),
            r.normal(0,np.sqrt(1/h),(dout,h)), np.zeros(dout)]
def net_fwd(p, x):
    W1,b1,W2,b2 = p; h = np.tanh(x@W1.T+b1); return h@W2.T+b2, (x,h)

target_net = init_net(4, 32, 8, seed=1)    # 固定随机，永不训练
pred_net   = init_net(4, 32, 8, seed=2)    # 训练去拟合 target

def rnd_bonus(s):
    t,_ = net_fwd(target_net, s[None,:]); p,_ = net_fwd(pred_net, s[None,:])
    return float(((t - p)**2).sum())

def adam_make(p): return [[np.zeros_like(w) for w in p],[np.zeros_like(w) for w in p],[0]]
def adam_step(p, grads, st, lr=1e-3):
    m,v,t = st; t[0]+=1
    for i,g in enumerate(grads):
        m[i]=0.9*m[i]+0.1*g; v[i]=0.999*v[i]+0.001*g*g
        p[i]-=lr*(m[i]/(1-0.9**t[0]))/(np.sqrt(v[i]/(1-0.999**t[0]))+1e-8)

seen = rng.uniform(-1, 1, (50, 4))          # 一簇见过的状态
novel = np.array([5.0, 5.0, 5.0, 5.0])      # 远处的新状态
b_seen0 = np.mean([rnd_bonus(s) for s in seen]); b_novel0 = rnd_bonus(novel)
# 训练预测网去拟合目标网(只在 seen 上)
st = adam_make(pred_net)
for _ in range(300):
    t,_ = net_fwd(target_net, seen); p,(x,h) = net_fwd(pred_net, seen)
    d = (p - t)/len(seen)
    dW2=d.T@h; db2=d.sum(0); dz=(d@pred_net[2])*(1-h**2); dW1=dz.T@x; db1=dz.sum(0)
    adam_step(pred_net, [dW1,db1,dW2,db2], st)
b_seen1 = np.mean([rnd_bonus(s) for s in seen]); b_novel1 = rnd_bonus(novel)
print(f'RND 内在奖励  见过状态: {b_seen0:.3f} -> {b_seen1:.3f} (训练后骤降)')
print(f'RND 内在奖励  新状态:   {b_novel0:.3f} -> {b_novel1:.3f} (仍很高)')
assert b_seen1 < b_seen0 * 0.5, '见过状态的内在奖励应随训练大幅下降'
assert b_novel1 > b_seen1 * 3, '新状态应保持高内在奖励'
print('✅ RND：见过的不新颖(误差→0)、没见过的新颖(误差高) -> 隐式伪计数驱动探索')

## 4 · 多智能体 RL：石头剪刀布收敛到混合纳什

RPS 无纯策略最优解，唯一纳什 = 均匀混合(各 1/3)。让两个**无悔学习者**(乘性权重更新)互相对打，
看它们的**平均策略**收敛到 1/3——学习动力学与博弈论均衡的美丽联系。

In [ ]:
# RPS 行玩家收益矩阵 A[i][j]: 石头0 剪刀1 布2
A = np.array([[0, 1, -1],
              [-1, 0, 1],
              [1, -1, 0]], dtype=float)   # 石头赢剪刀, 剪刀赢布, 布赢石头

def mw_update(w, payoff, lr=0.1):
    '''乘性权重(无悔学习): 收益高的动作权重指数增大，再归一化。'''
    w = w * np.exp(lr * payoff)
    return w / w.sum()

p1 = np.ones(3)/3; p2 = np.ones(3)/3       # 两玩家初始均匀
hist1 = []
for t in range(3000):
    payoff1 = A @ p2                         # 行玩家各动作对 p2 的期望收益
    payoff2 = (-A.T) @ p1                    # 列玩家收益(零和 = 负的行收益)
    p1 = mw_update(p1, payoff1, lr=0.05)
    p2 = mw_update(p2, payoff2, lr=0.05)
    hist1.append(p1.copy())
avg1 = np.mean(hist1[-1000:], axis=0)       # 平均策略(无悔学习收敛的是平均)
print('行玩家平均策略(后1000步):', np.round(avg1, 3), ' (纳什=均匀 0.333)')
assert np.allclose(avg1, 1/3, atol=0.05), '平均策略应收敛到均匀混合纳什'
print('✅ 两个无悔学习者在 RPS 中平均策略收敛到混合纳什 —— 学习⟷博弈均衡')

## 5 · Decision Transformer：按目标回报条件生成动作

DT 把 RL 变成序列建模：给 **return-to-go(目标回报)** 作条件，监督预测动作。
toy 版捕捉精髓：动作 a∈{0,1,2} 给奖励 = a。DT 学『RTG → 动作』，推理时**条件于高 RTG 输出高动作**。

In [ ]:
# 数据集：行为策略随机取动作，记录 (return-to-go, 动作)。1步任务 RTG = 奖励 = 动作值
data_rtg, data_act = [], []
for _ in range(3000):
    a = int(rng.integers(0, 3))
    data_rtg.append(float(a)); data_act.append(a)   # RTG = a (该动作的回报)
data_rtg = np.array(data_rtg)[:, None]; data_act = np.array(data_act)

def softmax(z): z = z - z.max(-1, keepdims=True); e = np.exp(z); return e/e.sum(-1, keepdims=True)

# 一个小 DT(MLP 版): 输入 RTG -> 动作 logits。捕捉 return-conditioning 精髓
dt = init_net(1, 32, 3, seed=5); st = adam_make(dt)
for _ in range(500):
    logits, (x, h) = net_fwd(dt, data_rtg); probs = softmax(logits)
    onehot = np.zeros_like(probs); onehot[np.arange(len(data_act)), data_act] = 1.0
    d = (probs - onehot)/len(data_act)          # 交叉熵梯度
    dW2=d.T@h; db2=d.sum(0); dz=(d@dt[2])*(1-h**2); dW1=dz.T@x; db1=dz.sum(0)
    adam_step(dt, [dW1,db1,dW2,db2], st, lr=3e-3)

# 推理：条件于不同目标回报，看 DT 输出的动作
for target_rtg in [0.0, 1.0, 2.0]:
    logits, _ = net_fwd(dt, np.array([[target_rtg]]))
    a = int(softmax(logits).argmax())
    print(f'条件于目标回报 RTG={target_rtg} -> DT 生成动作 {a}')
a_high = int(softmax(net_fwd(dt, np.array([[2.0]]))[0]).argmax())
a_low  = int(softmax(net_fwd(dt, np.array([[0.0]]))[0]).argmax())
assert a_high > a_low, '条件于更高目标回报应生成更高回报的动作'
print('✅ Decision Transformer return-conditioning：想要多少回报，就生成达到它的动作(纯监督学习!)')

---
## ✏️ 练习 1：count-based 探索奖励

实现 `count_intrinsic_reward(visit_counts, beta)`：给定每个状态的访问次数数组，返回每个状态的内在奖励 `β/√(N+1)`（+1 避免除零、也让未访问状态有界）。

In [ ]:
def count_intrinsic_reward(visit_counts, beta=1.0):
    # TODO: 返回 beta / sqrt(visit_counts + 1)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
counts = np.array([0, 1, 10, 100])
bonus = count_intrinsic_reward(counts, beta=2.0)
assert bonus.shape == (4,)
assert bonus[0] > bonus[1] > bonus[2] > bonus[3], '访问越少奖励越高'
assert np.isclose(bonus[0], 2.0), 'N=0: 2/sqrt(1)=2'
assert np.isclose(bonus[3], 2.0/np.sqrt(101)), 'N=100: 2/sqrt(101)'
print('count 内在奖励:', np.round(bonus, 3))
print('✅ 练习 1 通过：访问少的状态获得更高探索奖励')

## ✏️ 练习 2：RND 内在奖励

实现 `rnd_intrinsic(target_net, pred_net, states)`：给定一批状态 `(B, d)`，返回每个状态的 RND 内在奖励 `||g(s)-f(s)||²`（长度 B）。用上面的 `net_fwd`。

In [ ]:
def rnd_intrinsic(target_net, pred_net, states):
    # TODO: 前向 target 和 pred，返回每行 ||pred - target||^2
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
tn = init_net(4, 16, 8, seed=11); pn = init_net(4, 16, 8, seed=12)
S = rng.standard_normal((5, 4))
bonus = rnd_intrinsic(tn, pn, S)
assert bonus.shape == (5,)
assert np.all(bonus >= 0), '平方误差非负'
# 手工核对一行
t,_ = net_fwd(tn, S); p,_ = net_fwd(pn, S)
assert np.allclose(bonus, ((t - p)**2).sum(1)), '应等于逐行平方误差和'
print('RND 内在奖励(5个状态):', np.round(bonus, 3))
print('✅ 练习 2 通过：RND 内在奖励 = 预测网与目标网的平方差')

## ✏️ 练习 3：MARL 乘性权重更新

实现 `multiplicative_weights(weights, payoffs, lr)`：无悔学习的一步更新——`w ← w·exp(lr·payoff)` 再归一化为概率分布。返回新策略。

In [ ]:
def multiplicative_weights(weights, payoffs, lr=0.1):
    # TODO: w_new = weights * exp(lr * payoffs); 归一化(除以和)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
w = np.array([1/3, 1/3, 1/3])
payoffs = np.array([1.0, 0.0, -1.0])         # 动作0收益高
w2 = multiplicative_weights(w, payoffs, lr=0.5)
assert np.isclose(w2.sum(), 1.0), '应是概率分布'
assert w2[0] > w[0] and w2[2] < w[2], '高收益动作权重增、低收益减'
assert w2.argmax() == 0, '收益最高的动作权重最大'
print('更新后策略:', np.round(w2, 3))
print('✅ 练习 3 通过：乘性权重让高收益动作概率上升(无悔学习)')

## ✏️ 练习 4：Decision Transformer 的 return-to-go

实现 `returns_to_go(rewards)`：给定一条轨迹的逐步奖励 `[r0,r1,...]`，返回每步的 return-to-go `RTG_t = Σ_{k≥t} r_k`（从该步到结束的回报和）。这是 DT 的核心条件信号。

In [ ]:
def returns_to_go(rewards):
    # TODO: RTG_t = sum(rewards[t:]); 返回与 rewards 等长的数组(可用反向累加)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rewards = np.array([1.0, 2.0, 3.0, 4.0])
rtg = returns_to_go(rewards)
assert np.allclose(rtg, [10.0, 9.0, 7.0, 4.0]), 'RTG = 从该步到末尾的奖励和'
assert rtg[0] == rewards.sum(), '首步 RTG = 总回报'
assert rtg[-1] == rewards[-1], '末步 RTG = 最后一步奖励'
print('奖励:', rewards, '-> return-to-go:', rtg)
print('✅ 练习 4 通过：return-to-go 是 DT 条件生成的核心信号')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def count_intrinsic_reward(visit_counts, beta=1.0):
    return beta / np.sqrt(visit_counts + 1)

In [ ]:
# 练习 2
def rnd_intrinsic(target_net, pred_net, states):
    t, _ = net_fwd(target_net, states)
    p, _ = net_fwd(pred_net, states)
    return ((t - p) ** 2).sum(axis=1)

In [ ]:
# 练习 3
def multiplicative_weights(weights, payoffs, lr=0.1):
    w = weights * np.exp(lr * payoffs)
    return w / w.sum()

In [ ]:
# 练习 4
def returns_to_go(rewards):
    return np.cumsum(rewards[::-1])[::-1]

---
## 🧪 真实数据胶囊：硬探索基准与 DT 性能

**蒙特祖玛的复仇（Montezuma's Revenge）** 是硬探索的试金石——稀疏奖励 + 长动作链。下面是几类方法在它上面的**真实得分**（约数，公开论文），展示内在动机探索的威力。

In [ ]:
# Montezuma's Revenge 得分(约数；人类专家~4700)。展示探索方法的代差
MONTEZUMA = {
    'DQN (ε-greedy, 2015)':        0,        # ε-贪婪几乎拿不到分(稀疏奖励)
    'A3C (熵探索)':                  0,
    'Pseudo-count (Bellemare 2016)': 3439,    # 伪计数内在奖励
    'RND (Burda 2018)':            8152,      # 随机网络蒸馏(超人类!)
    'Go-Explore (2021)':           43000,     # 结构化探索
}
print(f"{'方法':34s}{'得分':>10s}")
for k, v in MONTEZUMA.items():
    print(f'{k:34s}{v:>10,}')
# 局部探索(ε-贪婪/熵)拿 0 分；内在动机(伪计数/RND)才突破
assert MONTEZUMA['DQN (ε-greedy, 2015)'] == 0, 'ε-贪婪在硬探索任务上拿0分'
assert MONTEZUMA['RND (Burda 2018)'] > 4000, 'RND 突破到超人类水平'
print('\n✅ 局部探索(ε/熵)在硬探索上彻底失败(0分)；内在动机(伪计数/RND/Go-Explore)才是出路')

**🧪 胶囊练习**：实现 `exploration_speedup(score_with, score_without)`：返回内在动机方法相对局部探索的得分提升。注意 `score_without` 可能为 0（用 `max(·,1)` 防除零），返回倍数。

In [ ]:
def exploration_speedup(score_with, score_without):
    # TODO: 返回 score_with / max(score_without, 1)
    raise NotImplementedError

In [ ]:
# 自测
g = exploration_speedup(MONTEZUMA['RND (Burda 2018)'], MONTEZUMA['DQN (ε-greedy, 2015)'])
assert g == 8152, 'RND 8152 vs DQN 0(防零->1) = 8152x'
print(f'RND 相对 ε-贪婪 提升 {g:.0f}x ✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def exploration_speedup(score_with, score_without):
    return score_with / max(score_without, 1)

### 小结
- **稀疏奖励**：要走对长链才有奖励时，ε-贪婪撞到的概率 `|A|^(-L)` 指数衰减 -> 局部探索失效。
- **内在奖励**：把『没见过/惊讶』变成可优化的奖励 `r = r_ext + β·r_int`：
  - **count-based**：`β/√N(s)`，访问少的给高奖励；
  - **RND**：对**固定随机目标网**的预测误差 = 新颖性(见过→误差低、新→误差高)，不怕 noisy-TV；
  - **好奇心 ICM**：预测自身动作后果的误差，逆模型滤无关噪声。
- **MARL**：环境含别的学习者 -> **非平稳**；合作(CTDE)、竞争(self-play/纳什)、一般和。无悔学习者在 RPS 收敛到混合纳什。
- **Decision Transformer**：把 RL 重铸成**序列建模**——`(RTG, s, a)` 序列、监督预测动作、推理时按目标回报条件生成。稳定简单，但拼接弱。

🎉 **恭喜你走完 C41 深度强化学习与决策！** 你已从零(纯 numpy)实现了 DQN、PPO、SAC 部件、CQL/IQL、MPC/Dyna、RND、Decision Transformer。

下一步：把这些验证过的训练循环搬上 **PyTorch + Gymnasium/MuJoCo + Stable-Baselines3/CleanRL**，换真实环境与规模。算法结构你已经懂了，剩下的是工程。